In [ ]:
import os
import kagglehub

os.environ['KAGGLEHUB_CACHE_DIR'] = "/content/kaggle_datasets"

download_path = kagglehub.dataset_download('tolgadincer/labeled-chest-xray-images')

print(download_path)

In [ ]:
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

# 1. 환경 설정
N_SPLITS = 5
path = '/root/.cache/kagglehub/datasets/tolgadincer/labeled-chest-xray-images/versions/1/chest_xray/'

train_dir = os.path.join(path, 'train')
test_dir = os.path.join(path, 'test')
categories = ['NORMAL', 'PNEUMONIA']

# 이진 분류(BCE)를 위한 라벨 맵핑 (출력 노드가 1개이므로 정상을 0, 폐렴을 1로 엄격히 정의)
label_to_idx = {'NORMAL': 0, 'PNEUMONIA': 1}

# 2. Train / Val 데이터프레임 빌드 (환자 ID 파싱 추가)
train_val_data = []
for category in categories:
    folder_path = os.path.join(train_dir, category)
    for img_name in os.listdir(folder_path):
        if img_name.endswith(('jpg', 'jpeg', 'png')):

            # 파일명 예시: 'person1000_bacteria_2931.jpeg' 또는 'IM-0115-0001.jpeg'
            # 1) 언더바(_)가 있으면 첫 번째 덩어리 추출 (예: 'person1000_bacteria_...jpeg' -> 'person1000')
            # 2) 언더바가 없고 하이픈(-)이 있으면 가장 오른쪽 하이픈 기준 왼쪽 전체 추출 (예: 'IM-0115-0001.jpeg' -> 'IM-0115')
            if '_' in img_name:
                patient_id = img_name.split('_')[0]
            elif '-' in img_name:
                patient_id = img_name.rsplit('-', 1)[0]
            else:
                patient_id = img_name.split('.')[0] # 그 외에는 확장자 떼고 파일명 통째로 ID 사용

            train_val_data.append({
                'image_path': os.path.join(folder_path, img_name),
                'label_name': category,
                'patient_id': patient_id # 그룹화를 위해 환자 ID 저장
            })

train_val_df = pd.DataFrame(train_val_data)
train_val_df['label'] = train_val_df['label_name'].map(label_to_idx)

# 3. Test 데이터프레임 빌드 (최종 채점을 위해 라벨 및 환자 ID 포함)
test_data = []
for category in categories:
    folder_path = os.path.join(test_dir, category)
    for img_name in os.listdir(folder_path):
        if img_name.endswith(('jpg', 'jpeg', 'png')):

            if '_' in img_name:
                patient_id = img_name.split('_')[0]
            elif '-' in img_name:
                patient_id = img_name.rsplit('-', 1)[0]
            else:
                patient_id = img_name.split('.')[0]

            test_data.append({
                'image_path': os.path.join(folder_path, img_name),
                'label_name': category,
                'patient_id': patient_id
            })

test_df = pd.DataFrame(test_data)
test_df['label'] = test_df['label_name'].map(label_to_idx)

# 4. StratifiedGroupKFold 적용
# 클래스 비율(y)도 맞추고, 같은 환자(groups)는 절대 찢어지지 않게 분할합니다.
train_val_df['fold'] = -1
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X=train_val_df, y=train_val_df['label'], groups=train_val_df['patient_id'])):
    train_val_df.loc[val_idx, 'fold'] = fold

# 데이터 검증 프린트
print(f"전체 Train/Val 이미지 수: {len(train_val_df)}장 | 고유 환자 수: {train_val_df['patient_id'].nunique()}명")
print(f"최종 Test 이미지 수: {len(test_df)}장\n")

# Fold별로 환자가 진짜 잘 격리되었는지 백엔드 검증 로그 출력
for fold in range(N_SPLITS):
    f_val = train_val_df[train_val_df['fold'] == fold]
    print(f"Fold {fold} -> 데이터: {len(f_val)}장 | 정상: {(f_val['label']==0).sum()}장, 폐렴: {(f_val['label']==1).sum()}장")

In [ ]:
import cv2
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

class ChestXrayBCEDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        self.df = df
        self.transform = transform
        self.is_test = is_test # 학습용/테스트용 구분을 위한 플래그

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 1. 이미지 로드 및 RGB 변환
        image = cv2.imread(row['image_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # 2. 이미지 증강(Transform) 적용
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        # 3. 💡 [BCE 핵심 포인트] 라벨 처리
        # BCEWithLogitsLoss는 라벨의 자료형이 FloatTensor여야 하고,
        # 차원이 [배치사이즈, 1]이어야 하므로, 여기서 차원을 늘릴 준비를 합니다.
        label = float(row['label'])
        label_tensor = torch.tensor([label], dtype=torch.float32) # 👈 [1] 형태로 감싸서 차원 확장

        return image, label_tensor

# 흑백 의료 영상의 특성을 고려한 안전한 데이터 증강 세팅 (색상 변형 금지)
train_transform = A.Compose([
    A.Resize(128, 128), # 배치 연산을 위한 고정 해상도
    A.HorizontalFlip(p=0.5), # 좌우 반전은 의학적으로 유효 (장기 위치 보존)
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_test_transform = A.Compose([
    A.Resize(128, 128),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import timm
from sklearn.metrics import f1_score # 클래스 불균형 채점을 위한 F1-Score 라이브러리
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
epochs = 3
BATCH_SIZE = 128
LR = 1e-3

test_dataset = ChestXrayBCEDataset(df=test_df, transform=val_test_transform)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# [앙상블 준비] 테스트 데이터 수만큼, 1개의 확률값을 누적할 빈 텐서 선언
total_test_preds = torch.zeros(len(test_df), 1).to(device)

for fold in range(N_SPLITS):
    print(f"\n==================== 🔥 FOLD {fold} START ====================")

    # 현재 Fold에 맞는 Train / Val 데이터 스플릿 및 DTO 매핑
    train_data = train_val_df[train_val_df['fold'] != fold].reset_index(drop=True)
    val_data = train_val_df[train_val_df['fold'] == fold].reset_index(drop=True)

    train_dataset = ChestXrayBCEDataset(df=train_data, transform=train_transform)
    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    val_dataset = ChestXrayBCEDataset(df=val_data, transform=val_test_transform)
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # [BCE 세팅 1] 모델 선언 (num_classes=1로 설정하여 1개의 단일 예측값 도출)
    model = timm.create_model('resnet18', pretrained=True, num_classes=1).to(device)

    # [BCE 세팅 2] 손실함수 교체 (CrossEntropyLoss -> BCEWithLogitsLoss)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    best_val_f1 = 0.0 # 클래스 불균형 대처를 위해 '최고의 F1-Score' 기준으로 가중치를 저장합니다!

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_preds_list = []
        train_labels_list = []

        for images, labels in train_dataloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images) # outputs shape: [배치사이즈, 1]
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)

            # [BCE 예측값 판정 로직]
            # outputs는 시그모이드를 거치기 전의 Logits 값입니다.
            # Logit이 0보다 크면 시그모이드 통과 시 0.5보다 커지므로 폐렴(1)으로 판정합니다.
            preds = (outputs > 0).float()

            # 평가지표(F1-Score) 계산을 위해 리스트에 누적 수집
            train_preds_list.extend(preds.cpu().numpy())
            train_labels_list.extend(labels.cpu().numpy())

        epoch_train_loss = train_loss / len(train_data)
        # sklearn을 활용한 Train 세트 F1-Score 연산
        epoch_train_f1 = f1_score(train_labels_list, train_preds_list, average='binary')

        model.eval()
        val_loss = 0.0
        val_preds_list = []
        val_labels_list = []

        with torch.no_grad():
            for images, labels in val_dataloader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)

                preds = (outputs > 0).float()
                val_preds_list.extend(preds.cpu().numpy())
                val_labels_list.extend(labels.cpu().numpy())

        epoch_val_loss = val_loss / len(val_data)
        epoch_val_f1 = f1_score(val_labels_list, val_preds_list, average='binary')

        print(f"Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f}, Train F1: {epoch_train_f1*100:.2f}%" \
               f" | Val Loss: {epoch_val_loss:.4f}, Val F1: {epoch_val_f1*100:.2f}%")

        if epoch_val_f1 > best_val_f1:
            best_val_f1 = epoch_val_f1
            torch.save(model.state_dict(), f"bce_best_model_fold{fold}.pth")

    print(f"--> Load Best Model of Fold {fold} (Val F1: {best_val_f1*100:.2f}%) and Predict Test Data")

    best_model = timm.create_model('resnet18', pretrained=True, num_classes=1).to(device)

    best_model.load_state_dict(torch.load(f"bce_best_model_fold{fold}.pth", map_location=device))
    best_model.eval()

    fold_test_preds = []
    with torch.no_grad():
        for images, _ in test_dataloader:
            images = images.to(device)
            outputs = best_model(images)

            # [BCE 확률 추출] 출력된 Logits 값에 시그모이드를 통과시켜 0.0~1.0 '폐렴일 확률'로 변환
            probs = torch.sigmoid(outputs)
            fold_test_preds.append(probs)

    # 앙상블을 위해 기존 빈 텐서 공간에 현재 Fold의 예측 확률 텐서를 차곡차곡 누적
    total_test_preds += torch.cat(fold_test_preds, 0)

# 집단 지성 평균 (Sort Voting Ensemble) 연산 및 최종 라벨 확정
final_test_probs = total_test_preds / N_SPLITS

# 최종 평균 확률이 0.5보다 크면 폐렴(1), 작으면 정상(0)으로 결정하여 넘파이 배열로 변환
final_classes = (final_test_probs > 0.5).int().cpu().numpy().flatten()

# 최종 결과를 DataFrame에 바인딩하고 파일로 내보내기
test_df['predicted_label'] = final_classes
test_df.to_csv('bce_submission.csv', index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

try:
    df = pd.read_csv('bce_submission.csv')
except FileNotFoundError:
    df = test_df.copy()

# 실제 정답(Ground Truth)과 모델의 예측(Prediction) 매핑
y_true = df['label'].values
y_pred = df['predicted_label'].values

# 사이킷런 엔진으로 Confusion Matrix 데이터 집계
cm = confusion_matrix(y_true, y_pred)

# 시각화 이쁘게 그리기 (Seaborn Heatmap)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['NORMAL (0)', 'PNEUMONIA (1)'],
            yticklabels=['NORMAL (0)', 'PNEUMONIA (1)'])

plt.title('Chest X-ray BCE 5-Fold Ensemble Confusion Matrix')
plt.ylabel('Actual Label (True)')
plt.xlabel('Predicted Label (Model)')
plt.show()

print(classification_report(y_true, y_pred, target_names=['NORMAL', 'PNEUMONIA']))